# ATF6 + EIF2AK3 + ERN1 perturbation analysis

This notebook contains the final reproducible workflow for the three-gene perturbation case:

1. optionally export per-cell BaiZe predictions from a trained checkpoint;
2. summarize observed and predicted expression responses;
3. generate a delta heatmap;
4. generate a full-expression bar plot with SEM error bars.

Exploratory diagnostics, repeated style revisions, and superseded plotting cells have been removed.


In [1]:
import argparse
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.patches import Rectangle
from tqdm.auto import tqdm

REPO_DIR = Path("/root/autodl-tmp/scERso")
DATA_PATH = Path(
    "/root/autodl-fs/triple_perturb/adamson_upr/processed/"
    "adamson_upr_triple_clean_split_hvg5000_log1p_dense.h5ad"
)
MODEL_PATH = REPO_DIR / (
    "adamson_upr_triple/checkpoints_hvg5000_dense_final/"
    "best_model_diff.pth"
)
PREDICTION_NPZ = REPO_DIR / (
    "adamson_upr_triple/predictions_best_diff/"
    "ATF6__EIF2AK3__ERN1_percell_pred_true_ctrl_residual00.npz"
)
OUTPUT_DIR = REPO_DIR / (
    "adamson_upr_triple/figures/three_gene_paper_palette"
)

PERTURBATION = "ATF6+EIF2AK3+ERN1"
PERTURBATION_STUB = "ATF6_EIF2AK3_ERN1"

CURATED_GENES = [
    "SDF2L1", "HSPA5", "HERPUD1", "DDIT3",
    "HSPA8", "MANF", "HSP90B1", "DDIT4",
    "TRIB3", "DNAJB11", "SELK", "XBP1",
]

TOP_N = 12
RUN_INFERENCE = False
BATCH_SIZE = 256
SAMPLE_STEPS = 50
GUIDANCE_SCALE = 1.0
RESIDUAL_SAMPLE_SCALE = 0.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

COLOR_PAIRS = {
    "rose": {"dark": "#8F1F22", "light": "#BF888E"},
    "blue": {"dark": "#004E85", "light": "#7EA4C2"},
    "teal": {"dark": "#107A83", "light": "#7DB7BE"},
}

GROUP_COLORS = {
    "Control": COLOR_PAIRS["blue"],
    "Predicted": COLOR_PAIRS["rose"],
    "Observed": COLOR_PAIRS["teal"],
}

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.linewidth": 0.8,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 8,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

print(f"Repository: {REPO_DIR}")
print(f"Prediction file: {PREDICTION_NPZ}")
print(f"Output directory: {OUTPUT_DIR}")


Repository: /root/autodl-tmp/scERso
Prediction file: /root/autodl-tmp/scERso/adamson_upr_triple/predictions_best_diff/ATF6__EIF2AK3__ERN1_percell_pred_true_ctrl_residual00.npz
Output directory: /root/autodl-tmp/scERso/adamson_upr_triple/figures/three_gene_paper_palette


In [ ]:
def export_per_cell_predictions() -> None:
    from utils.data_processor import DataProcessor
    import evaluate_diffusion as evaluation

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=False,
    )

    evaluation.get_args_cache = argparse.Namespace(
        pretrained_emb=None,
        scgpt_gene_emb_path=None,
        gene_prior_scale=0.1,
        target_mode="delta",
    )

    processor = DataProcessor(
        str(DATA_PATH),
        split_strategy="custom",
        split_col="split",
        perturb_parse_mode="raw",
        background_key="cell_line",
    )
    n_genes, n_perturbations, _ = processor.load_data()

    _, _, test_loader = processor.prepare_loaders(
        batch_size=BATCH_SIZE,
        rna_noise=0.0,
        control_match_scope="global",
        control_prototype_mode="topk_weighted",
        control_match_k=32,
    )

    model = evaluation.load_model_from_checkpoint(
        checkpoint=checkpoint,
        n_genes=n_genes,
        n_perts=n_perturbations,
        processor=processor,
        device=device,
        target_mode_override="delta",
    )
    model.eval()
    model.residual_sample_scale = RESIDUAL_SAMPLE_SCALE

    drug_embeddings = (
        processor.drug_embeddings.to(device)
        if processor.drug_embeddings is not None
        else None
    )

    predictions = []
    observed = []
    controls = []
    perturbation_ids = []

    with torch.no_grad():
        for batch in tqdm(
            test_loader,
            desc="Exporting per-cell predictions",
        ):
            control = batch["rna_control"].to(device)
            target = batch["rna_target"].to(device)
            perturbation = batch["perturb"].to(device)

            optional_inputs = {
                name: batch[name].to(device) if name in batch else None
                for name in [
                    "perturb_gene_idx",
                    "is_control",
                    "condition_id",
                    "source_flag",
                    "dose",
                    "atac_feat",
                ]
            }

            drug_features = (
                drug_embeddings[perturbation]
                if drug_embeddings is not None
                else None
            )

            prediction = model.predict_single(
                rna_control=control,
                perturb=perturbation,
                dose=optional_inputs["dose"],
                atac_feat=optional_inputs["atac_feat"],
                drug_feat=drug_features,
                sample_steps=SAMPLE_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                perturb_gene_idx=optional_inputs["perturb_gene_idx"],
                is_control=optional_inputs["is_control"],
                condition_id=optional_inputs["condition_id"],
                source_flag=optional_inputs["source_flag"],
            )

            predictions.append(prediction.detach().cpu().numpy())
            observed.append(target.detach().cpu().numpy())
            controls.append(control.detach().cpu().numpy())
            perturbation_ids.append(
                batch["perturb"].detach().cpu().numpy()
            )

    prediction_matrix = np.concatenate(predictions, axis=0)
    observed_matrix = np.concatenate(observed, axis=0)
    control_matrix = np.concatenate(controls, axis=0)
    perturbation_ids = np.concatenate(perturbation_ids, axis=0)

    gene_names = np.asarray(
        processor.adata.var_names.tolist(),
        dtype=str,
    )
    perturbation_names = np.asarray(
        [
            processor.id_to_perturb[int(index)]
            for index in perturbation_ids
        ],
        dtype=str,
    )

    PREDICTION_NPZ.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        PREDICTION_NPZ,
        pred=prediction_matrix.astype(np.float32),
        true=observed_matrix.astype(np.float32),
        ctrl=control_matrix.astype(np.float32),
        gene_names=gene_names,
        perturb_ids=perturbation_ids,
        perturb_names=perturbation_names,
    )

    print(f"Saved predictions to: {PREDICTION_NPZ}")
    print(f"Prediction shape: {prediction_matrix.shape}")


if RUN_INFERENCE or not PREDICTION_NPZ.exists():
    export_per_cell_predictions()
else:
    print(f"Using existing predictions: {PREDICTION_NPZ}")


In [ ]:
def standard_error(matrix: np.ndarray) -> np.ndarray:
    if matrix.shape[0] < 2:
        return np.zeros(matrix.shape[1], dtype=float)
    return matrix.std(axis=0, ddof=1) / np.sqrt(matrix.shape[0])


def is_unwanted_gene(gene: str) -> bool:
    gene = str(gene).upper()
    unwanted_prefixes = (
        "RPL", "RPS", "MT-", "MTRNR",
        "HBA", "HBB", "HBG", "HBM", "HBQ", "HBZ",
    )
    unwanted_exact = {"FTL", "FTH1", "MALAT1", "NEAT1", "B2M"}
    return gene in unwanted_exact or gene.startswith(unwanted_prefixes)


data = np.load(PREDICTION_NPZ, allow_pickle=True)
prediction_matrix = data["pred"]
observed_matrix = data["true"]
control_matrix = data["ctrl"]
gene_names = data["gene_names"].astype(str)

if "perturb_names" in data.files:
    perturbation_names = data["perturb_names"].astype(str)
    target_mask = perturbation_names == PERTURBATION
    if not target_mask.any():
        available = sorted(set(perturbation_names.tolist()))
        raise ValueError(
            f"Perturbation '{PERTURBATION}' was not found. "
            f"Available perturbations include: {available[:20]}"
        )
    prediction_matrix = prediction_matrix[target_mask]
    observed_matrix = observed_matrix[target_mask]
    control_matrix = control_matrix[target_mask]

if not (
    prediction_matrix.shape
    == observed_matrix.shape
    == control_matrix.shape
):
    raise ValueError("Prediction, observed, and control matrices must match.")

gene_to_index = {
    gene: index for index, gene in enumerate(gene_names)
}

control_mean_all = control_matrix.mean(axis=0)
predicted_mean_all = prediction_matrix.mean(axis=0)
observed_mean_all = observed_matrix.mean(axis=0)
observed_delta_all = observed_mean_all - control_mean_all

selected_genes = [
    gene for gene in CURATED_GENES if gene in gene_to_index
]

if len(selected_genes) < TOP_N:
    ranked_indices = np.argsort(np.abs(observed_delta_all))[::-1]
    for index in ranked_indices:
        gene = gene_names[index]
        if gene in selected_genes or is_unwanted_gene(gene):
            continue
        selected_genes.append(gene)
        if len(selected_genes) == TOP_N:
            break

selected_genes = selected_genes[:TOP_N]
selected_indices = [gene_to_index[gene] for gene in selected_genes]

control_selected = control_matrix[:, selected_indices]
predicted_selected = prediction_matrix[:, selected_indices]
observed_selected = observed_matrix[:, selected_indices]

control_mean = control_selected.mean(axis=0)
predicted_mean = predicted_selected.mean(axis=0)
observed_mean = observed_selected.mean(axis=0)

control_sem = standard_error(control_selected)
predicted_sem = standard_error(predicted_selected)
observed_sem = standard_error(observed_selected)

observed_delta = observed_mean - control_mean
predicted_delta = predicted_mean - control_mean
delta_error = predicted_delta - observed_delta

summary = pd.DataFrame({
    "gene": selected_genes,
    "control_mean": control_mean,
    "control_sem": control_sem,
    "predicted_mean": predicted_mean,
    "predicted_sem": predicted_sem,
    "observed_mean": observed_mean,
    "observed_sem": observed_sem,
    "observed_delta": observed_delta,
    "predicted_delta": predicted_delta,
    "delta_error": delta_error,
})

summary_path = (
    OUTPUT_DIR
    / f"{PERTURBATION_STUB}_top{TOP_N}_summary.csv"
)
summary.to_csv(summary_path, index=False)

print(f"Cells used: {prediction_matrix.shape[0]}")
print(f"Genes used: {prediction_matrix.shape[1]}")
print(f"Saved summary: {summary_path}")
display(summary.round(4))


In [ ]:
heatmap_values = summary[
    ["observed_delta", "predicted_delta", "delta_error"]
].to_numpy()

vmax = max(1.0, float(np.nanmax(np.abs(heatmap_values))))
vmax = np.ceil(vmax * 10) / 10
vmin = -vmax

heatmap_cmap = LinearSegmentedColormap.from_list(
    "baize_delta",
    [
        COLOR_PAIRS["blue"]["dark"],
        COLOR_PAIRS["blue"]["light"],
        "#F5F2EF",
        COLOR_PAIRS["rose"]["light"],
        COLOR_PAIRS["rose"]["dark"],
    ],
    N=256,
)
norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

figure_height = max(4.6, 0.36 * len(selected_genes) + 1.6)
fig = plt.figure(figsize=(5.8, figure_height))
ax = fig.add_axes([0.22, 0.14, 0.50, 0.70])

image = ax.imshow(
    heatmap_values,
    aspect="auto",
    cmap=heatmap_cmap,
    norm=norm,
    interpolation="nearest",
)

for row in range(heatmap_values.shape[0]):
    for column in range(heatmap_values.shape[1]):
        value = heatmap_values[row, column]
        color = "white" if abs(value) >= 0.55 * vmax else "black"
        ax.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=7.2,
            color=color,
        )

ax.set_xticks(np.arange(3))
ax.set_xticklabels(
    ["Observed", "Predicted", "Error"],
    rotation=40,
    ha="right",
)
ax.set_yticks(np.arange(len(selected_genes)))
ax.set_yticklabels(selected_genes)
ax.tick_params(axis="both", length=0)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_color("0.25")

method_colors = [
    GROUP_COLORS["Observed"]["dark"],
    GROUP_COLORS["Predicted"]["dark"],
    GROUP_COLORS["Control"]["dark"],
]
top_bar = fig.add_axes([0.22, 0.852, 0.50, 0.028])
top_bar.set_xlim(-0.5, 2.5)
top_bar.set_ylim(0, 1)
top_bar.axis("off")
for column, color in enumerate(method_colors):
    top_bar.add_patch(
        Rectangle(
            (column - 0.5, 0),
            1,
            1,
            facecolor=color,
            edgecolor="none",
        )
    )

colorbar_ax = fig.add_axes([0.76, 0.14, 0.035, 0.70])
colorbar = fig.colorbar(image, cax=colorbar_ax)
colorbar.set_label("Expression delta")

fig.suptitle(
    f"{PERTURBATION}: observed and predicted expression deltas",
    fontsize=11,
    y=0.97,
)

heatmap_base = (
    OUTPUT_DIR
    / f"{PERTURBATION_STUB}_top{TOP_N}_delta_heatmap"
)
fig.savefig(
    heatmap_base.with_suffix(".png"),
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    heatmap_base.with_suffix(".svg"),
    bbox_inches="tight",
)
fig.savefig(
    heatmap_base.with_suffix(".pdf"),
    bbox_inches="tight",
)
plt.show()

print(f"Saved heatmap files with prefix: {heatmap_base}")


In [ ]:
x = np.arange(len(selected_genes))
bar_width = 0.24

fig, ax = plt.subplots(figsize=(11.4, 4.8))

series = [
    (
        "Control",
        control_mean,
        control_sem,
        x - bar_width,
    ),
    (
        "Predicted",
        predicted_mean,
        predicted_sem,
        x,
    ),
    (
        "Observed",
        observed_mean,
        observed_sem,
        x + bar_width,
    ),
]

bar_handles = []
for label, means, errors, positions in series:
    colors = GROUP_COLORS[label]
    bars = ax.bar(
        positions,
        means,
        width=bar_width,
        color=colors["light"],
        edgecolor=colors["dark"],
        linewidth=0.7,
        label=label,
        zorder=2,
    )
    ax.errorbar(
        positions,
        means,
        yerr=errors,
        fmt="none",
        ecolor=colors["dark"],
        elinewidth=1.1,
        capsize=2.5,
        capthick=1.1,
        zorder=3,
    )
    bar_handles.append(bars)

ax.set_xticks(x)
ax.set_xticklabels(
    selected_genes,
    rotation=55,
    ha="right",
)
ax.set_ylabel("Expression value")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.grid(
    True,
    linestyle="--",
    linewidth=0.5,
    alpha=0.28,
)
ax.set_axisbelow(True)

upper_bound = max(
    np.max(control_mean + control_sem),
    np.max(predicted_mean + predicted_sem),
    np.max(observed_mean + observed_sem),
)
ax.set_ylim(0, upper_bound * 1.20)

fig.suptitle(
    f"Expression of selected genes after {PERTURBATION} perturbation",
    fontsize=11,
    y=0.985,
)
fig.legend(
    handles=bar_handles,
    labels=["Control", "Predicted", "Observed"],
    frameon=False,
    ncol=3,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.925),
)

plt.subplots_adjust(
    top=0.78,
    bottom=0.25,
    left=0.08,
    right=0.98,
)

barplot_base = (
    OUTPUT_DIR
    / f"{PERTURBATION_STUB}_top{TOP_N}_expression_barplot"
)
fig.savefig(
    barplot_base.with_suffix(".png"),
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    barplot_base.with_suffix(".svg"),
    bbox_inches="tight",
)
fig.savefig(
    barplot_base.with_suffix(".pdf"),
    bbox_inches="tight",
)
plt.show()

print(f"Saved bar plot files with prefix: {barplot_base}")
